In [ ]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import START, END, StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
class MessageState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]



llm = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest")

In [ ]:
def chat_node(state: MessageState):

    # take user query from state
    messages = state['messages']

    # send query to llm
    response = llm.invoke(messages)

    # store response to state
    return {"messages": [response]}

In [ ]:
checkpointer = MemorySaver()        # where to store state of graph when end
graph = StateGraph(MessageState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
thread_id = '1'

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "exit", "bye"]:
        break

    config = {"configurable": {"thread_id": thread_id}}
    response = chatbot.invoke({"messages": HumanMessage(content=user_input)}, config=config)

    print("You:", user_input)
    print("AI:", response["messages"][-1].text)

### See History

In [ ]:
chatbot.get_state(config=config)